# PN20 — one-rung, two-child prime-location development audit

This notebook reproduces the three development translations and the final branch-aware table. The supplied 87-bit anchor remains sealed and is not present in the notebook. All target labels below had already been opened before PN20.


In [1]:
import json
from pathlib import Path

HERE = Path.cwd()
numeric = json.loads((HERE / 'PN20_ONE_RUNG_DEVELOPMENT.json').read_text(encoding='utf-8'))
directional = json.loads((HERE / 'PN20_DIRECTIONAL_TWO_CHILD_DEVELOPMENT.json').read_text(encoding='utf-8'))
branch = json.loads((HERE / 'PN20_BRANCH_TWO_CHILD_DEVELOPMENT.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN20_ONE_RUNG_TWO_CHILD_VALIDATION.json').read_text(encoding='utf-8'))
print('Loaded three development artifacts and independent validation.')


Loaded three development artifacts and independent validation.


## Outcome

The important number is the exact next-prime count, not visual closeness to the normalized ridge. A method intended to find the next prime must return its integer location.


In [2]:
print('numerical-largest formulas:')
for name, summary in numeric['formula_summary'].items():
    print(f"  {name}: exact {summary['exact_count']}/{summary['anchor_count']}")
print('unrestricted directional:', directional['summary'])
print('branch-aware:', branch['summary'])
assert directional['summary']['exact_next_primes'] == 0
assert branch['summary']['exact_next_primes'] == 0


numerical-largest formulas:
  three_minus_ab: exact 0/7
  two_ab_over_two_minus_ab_plus_one: exact 1/7
  two_ab_over_parenthesized_ba_plus_one: exact 0/7
  secant_n_to_2n: exact 0/7
  raw_ab_deficit: exact 0/7
unrestricted directional: {'anchor_count': 7, 'forward_predictions': 4, 'exact_next_primes': 0, 'prime_predictions': 0, 'mean_absolute_integer_error': 1870051659823.1428}
branch-aware: {'anchor_count': 7, 'two_branch_states_at_both_landmarks': 7, 'forward_predictions': 3, 'exact_next_primes': 0, 'prime_predictions': 1, 'mean_absolute_integer_error': 638907342447.3334}


## Branch-aware two-child states

The retained A child is the immediate child furthest toward the ridge from 0. The retained B child is the immediate child furthest toward the ridge from 2. `AB` is their mean progress; `BA=2-AB`.


In [3]:
header = ('anchor', 'AB(N)', 'AB(2N)', 'prediction', 'actual next prime', 'exact')
print(' | '.join(header))
for row in branch['rows']:
    values = (
        str(row['anchor']),
        f"{row['landmark_1']['phase_ab']:.9f}",
        f"{row['landmark_2']['phase_ab']:.9f}",
        str(row['predicted_integer']),
        str(row['true_next_prime']),
        str(row['exact_next_prime']),
    )
    print(' | '.join(values))
    assert abs(row['landmark_1']['phase_ab'] + row['landmark_1']['phase_ba'] - 2.0) < 1e-12


anchor | AB(N) | AB(2N) | prediction | actual next prime | exact
100000000 | 0.704053279 | 0.784805664 | 466486663 | 100000007 | False
1000000000 | 0.976102986 | 0.742818116 | None | 1000000007 | False
10000000000 | 0.834638835 | 0.663380549 | None | 10000000019 | False
100000000000 | 0.653750816 | 0.642342601 | None | 100000000003 | False
400000000000 | 0.703787490 | 0.845852282 | 1234020887773 | 400000000019 | False
700000000000 | 0.830213884 | 0.940023065 | 1782334652941 | 700000000009 | False
900000000000 | 0.870863245 | 0.640615379 | None | 900000000013 | False


## Algebraic collapse of the proposed confirmation expression

Under ordinary precedence, `2*AB/2 - AB + 1` is identically one. It verifies normalization but cannot carry a large integer location.


In [4]:
for ab in (0.1, 0.5, 1.0, 1.5, 1.9):
    value = 2.0 * ab / 2.0 - ab + 1.0
    print(ab, value)
    assert abs(value - 1.0) < 1e-12


0.1 1.0
0.5 1.0
1.0 1.0
1.5 1.0
1.9 1.0


## Independent validation

The validator uses its own bytearray sieve and trial-division prime check. It independently recomputes the immediate children, the next-prime truth labels and all declared exact counts.


In [5]:
print(validation['status'], validation['checks_passed'], '/', validation['checks_total'])
for item in validation['checks']:
    print('PASS' if item['passed'] else 'FAIL', '-', item['label'])
assert validation['status'] == 'PASS'
assert validation['checks_passed'] == validation['checks_total']


PASS 12 / 12
PASS - seven development anchors
PASS - all three artifacts declare development only
PASS - branch summary exact count independently visible
PASS - directional summary exact count
PASS - numeric formulas never exact except constant-three variant once
PASS - independent branch states match
PASS - AB plus BA equals pure TE-ARA two
PASS - independent next-prime labels match
PASS - independent exact prediction count
PASS - independent prime prediction count
PASS - written confirmation expression is identity
PASS - no raw 25-plus-digit fresh anchor in PN20 artifacts


## Interpretation

This is a null for the literal two-scalar location decoder, not for ARA generally. The two children may describe a local identity, but compressing them to `1` and `1` removes scale and produces a many-to-one map. A future one-rung sufficient statistic must retain a non-collapsing location coordinate and must count the cost of selecting its two children.
